In [1]:
# ATTENDANCE LOGIC

In [ ]:
import urllib.request
import cv2
import numpy as np
from keras.models import load_model
from datetime import datetime
import csv
import os
import time


# ============================================================
# 1. LOAD HAAR CASCADE
# ============================================================

classifier = cv2.CascadeClassifier(
    r"C:\Users\QS\Desktop\Face_ Recognition\project_face_detect\haarcascade_frontalface_default.xml"
)


# ============================================================
# 2. LOAD TRAINED CNN MODEL
# ============================================================

model = load_model(
   r"C:\Users\QS\Desktop\Face_ Recognition\project_face_detect\final_model.h5"
)


# ============================================================
# 3. IP CAMERA URL
# ============================================================

URL = "http://192.168.1.5:8080/shot.jpg"


# ============================================================
# 4. PERSON LABELS
# IMPORTANT: ORDER MUST MATCH YOUR CNN TRAINING LABELS
# ============================================================

labels = [
    "Alisha",
    "Anwesha",
    "Bhumika",
    "Mumma"
]


# ============================================================
# 5. ATTENDANCE DICTIONARY
# ============================================================

attendance = {}

for name in labels:

    attendance[name] = {
        "entry": None,
        "exit": None
    }


# ============================================================
# 6. PREPROCESS FUNCTION
# ============================================================

def preprocess(img):

    # Convert BGR to grayscale
    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    # Resize according to trained model
    img = cv2.resize(
        img,
        (100, 100)
    )

    # Histogram equalization
    img = cv2.equalizeHist(
        img
    )

    # Reshape
    img = img.reshape(
        1,
        100,
        100,
        1
    )

    # Normalize
    img = img / 255.0

    return img


# ============================================================
# 7. GET PREDICTED PERSON
# ============================================================

def get_pred_label(pred):

    return labels[pred]


# ============================================================
# 8. CREATE ATTENDANCE CSV FILE
# ============================================================

csv_file = "attendance.csv"


# ============================================================
# 9. START IP CAMERA
# ============================================================

print("Starting IP camera...")

print("Press Q to stop attendance system.")


# Keep track of people already recognized
recognized_people = set()


# ============================================================
# 10. MAIN CAMERA LOOP
# ============================================================

while True:

    try:

        # Get image from IP camera
        img_resp = urllib.request.urlopen(
            URL,
            timeout=5
        )

        # Convert image to NumPy array
        imgnp = np.array(
            bytearray(
                img_resp.read()
            ),
            dtype=np.uint8
        )

        # Decode image
        frame = cv2.imdecode(
            imgnp,
            -1
        )

        if frame is None:

            print(
                "Could not read image from camera."
            )

            continue


        # ====================================================
        # CONVERT TO GRAYSCALE FOR HAAR CASCADE
        # ====================================================

        gray = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )


        # ====================================================
        # DETECT FACES
        # ====================================================

        faces = classifier.detectMultiScale(
            gray,
            scaleFactor=1.3,
            minNeighbors=5,
            minSize=(50, 50)
        )


        # ====================================================
        # PROCESS EACH FACE
        # ====================================================

        for (
            x,
            y,
            w,
            h
        ) in faces:


            # Crop face
            face = frame[
                y:y+h,
                x:x+w
            ]


            # Preprocess face
            processed_face = preprocess(
                face
            )


            # =================================================
            # CNN PREDICTION
            # =================================================

            prediction = model.predict(
                processed_face,
                verbose=0
            )


            # Get predicted index
            pred_index = np.argmax(
                prediction[0]
            )


            # Get confidence
            confidence = (
                prediction[0][pred_index]
                * 100
            )


            # Get person name
            person_name = get_pred_label(
                pred_index
            )


            # =================================================
            # CURRENT TIME
            # =================================================

            current_time = datetime.now()


            # =================================================
            # RECORD ENTRY TIME
            # =================================================

            if (
                person_name
                not in recognized_people
                and confidence >= 70
            ):

                attendance[
                    person_name
                ][
                    "entry"
                ] = current_time

                recognized_people.add(
                    person_name
                )

                print(
                    f"{person_name} entered at "
                    f"{current_time.strftime('%H:%M:%S')}"
                )


            # =================================================
            # DISPLAY FACE RECTANGLE
            # =================================================

            cv2.rectangle(
                frame,
                (x, y),
                (x+w, y+h),
                (0, 255, 0),
                2
            )


            # =================================================
            # DISPLAY NAME AND CONFIDENCE
            # =================================================

            text = (
                f"{person_name} "
                f"{confidence:.1f}%"
            )


            cv2.putText(
                frame,
                text,
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )


        # ====================================================
        # DISPLAY CAMERA
        # ====================================================

        cv2.imshow(
            "Face Recognition Attendance",
            frame
        )


        # ====================================================
        # PRESS Q TO STOP
        # ====================================================

        key = cv2.waitKey(
            1
        ) & 0xFF


        if key == ord("q"):

            print(
                "\nStopping attendance system..."
            )

            break


    except Exception as e:

        print(
            "Camera error:",
            e
        )

        time.sleep(
            1
        )


# ============================================================
# 11. RELEASE CAMERA
# ============================================================

cv2.destroyAllWindows()


# ============================================================
# 12. RECORD EXIT TIME
# ============================================================

exit_time = datetime.now()


for person_name in recognized_people:

    attendance[
        person_name
    ][
        "exit"
    ] = exit_time


# ============================================================
# 13. CALCULATE TOTAL HOURS
# ============================================================

for person_name in recognized_people:

    entry = attendance[
        person_name
    ][
        "entry"
    ]

    exit = attendance[
        person_name
    ][
        "exit"
    ]


    if (
        entry is not None
        and exit is not None
    ):

        total_time = (
            exit - entry
        )


        total_seconds = int(
            total_time.total_seconds()
        )


        hours = (
            total_seconds // 3600
        )


        minutes = (
            (total_seconds % 3600)
            // 60
        )


        seconds = (
            total_seconds % 60
        )


        attendance[
            person_name
        ][
            "total_hours"
        ] = (
            f"{hours:02d}:"
            f"{minutes:02d}:"
            f"{seconds:02d}"
        )


# ============================================================
# 14. SAVE ATTENDANCE TO CSV
# ============================================================

with open(
    csv_file,
    "w",
    newline=""
) as file:

    writer = csv.writer(
        file
    )


    # Header
    writer.writerow(
        [
            "Name",
            "Entry Time",
            "Exit Time",
            "Total Hours"
        ]
    )


    # Write records
    for person_name in labels:

        if person_name in recognized_people:

            entry = attendance[
                person_name
            ][
                "entry"
            ]


            exit = attendance[
                person_name
            ][
                "exit"
            ]


            total = attendance[
                person_name
            ][
                "total_hours"
            ]


            writer.writerow(
                [
                    person_name,
                    entry.strftime(
                        "%Y-%m-%d %H:%M:%S"
                    ),
                    exit.strftime(
                        "%Y-%m-%d %H:%M:%S"
                    ),
                    total
                ]
            )


# ============================================================
# 15. PRINT FINAL ATTENDANCE
# ============================================================

print(
    "\n========== ATTENDANCE REPORT =========="
)


for person_name in labels:

    if person_name in recognized_people:

        print(
            "\nName:",
            person_name
        )

        print(
            "Entry Time:",
            attendance[
                person_name
            ][
                "entry"
            ].strftime(
                "%H:%M:%S"
            )
        )

        print(
            "Exit Time:",
            attendance[
                person_name
            ][
                "exit"
            ].strftime(
                "%H:%M:%S"
            )
        )

        print(
            "Total Hours:",
            attendance[
                person_name
            ][
                "total_hours"
            ]
        )

    else:

        print(
            "\nName:",
            person_name,
            "-> Absent"
        )


print(
    "\nAttendance saved to:",
    csv_file
)

Starting IP camera...
Press Q to stop attendance system.
Anwesha entered at 22:49:08
Alisha entered at 22:49:08
Camera error: <urlopen error timed out>
Camera error: <urlopen error timed out>
Bhumika entered at 22:49:47
Camera error: <urlopen error timed out>
Camera error: <urlopen error timed out>
Camera error: <urlopen error timed out>
Camera error: <urlopen error timed out>
Camera error: <urlopen error timed out>
Camera error: <urlopen error timed out>
Camera error: <urlopen error timed out>
Camera error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\imgcodecs\src\loadsave.cpp:993: error: (-215:Assertion failed) !buf.empty() in function 'cv::imdecode_'

Camera error: <urlopen error timed out>
Camera error: <urlopen error timed out>
Camera error: <urlopen error timed out>
Camera error: <urlopen error timed out>
Camera error: <urlopen error [WinError 10061] No connection could be made because the target machine actively refused it>
Camera error: <urlopen error [WinErr